In [1]:
# %cd ~/repos/foo_cleaned

In [2]:
import os
import numpy as np
import pybamm
from scipy.stats import qmc
import src.util.functions as functions
import multiprocessing as mp

In [3]:
print(os.cpu_count())            # logical cores available to the process
print(mp.cpu_count())  # same thing

48
48


In [4]:
def get_soc_and_diffusivities_sobol(
    soc_levels,
    total_samples,
    log_lower=-14,
    log_upper=-12,
    rng = None
):
    n_soc = len(soc_levels)
    sampler   = qmc.Sobol(d=3, scramble=True, seed=rng) # We use 3 dimensions for the Sobol sequence
    u         = sampler.random(total_samples) # We sample all points in the socxDanxDca domain
    soc_levels = np.asarray(soc_levels)
    soc_idx   = (u[:, 0] * n_soc).astype(int)   # we scale the first column to the number of SoC levels
    soc_col   = soc_levels[soc_idx]             # we get the SoC levels from the first column
    log_cols  = qmc.scale(u[:, 1:], log_lower, log_upper) # we scale the other two columns to the log_diffusion range

    results   = np.column_stack((soc_col, log_cols)) # we stack the columns together

    return results

In [5]:
def PLSCurrent(C, t_max, rng=np.random.default_rng()):
    """
    Return I_func(t) that produces a rectangular-pulse current profile.

    • pulses_per_hour ∈ [1, 10]   (uniform)
    • All pulses in one sample share ONE peak: ±(0.5‒1)·C.
    • Width is a fixed fraction (5‒30 %) of the period → typically 2-30 s.
    """
    pulses_per_hour = rng.integers(1, 11)                    # 1–10 pulses / h
    n_pulses = max(1, int(pulses_per_hour * t_max / 3600))   # fit into t_max

    direction = rng.choice([1, -1])                          # charge or discharge
    peak = direction * C * rng.uniform(0.5, 1.0)             # fixed amplitude

    period = t_max / n_pulses
    duty_cycle = rng.uniform(0.2, 0.7)                     # 10–30 %
    pulse_width = period * duty_cycle                        # hold-time in s

    starts = np.arange(n_pulses) * period                    # equally spaced

    def I_func(t):
        t = np.asarray(t)
        I = np.zeros_like(t, dtype=float)
        for st in starts:
            on = (t >= st) & (t < st + pulse_width)          # rectangular gate
            I[on] = peak
        return I

    return I_func

In [6]:
def generate_data(key, num, family="CC"):
    """
    Generate a list of current profiles using NumPy for random numbers.
    
    The seed is used to ensure reproducibility.
    """
    rng = np.random.default_rng(key)

    I_list = []
    if family == "GRF":
        for _ in range(num):
            s = rng.integers(0, 2**31 - 1)
            I_func = functions.GaussianRFCurrent(s, C, t_max)
            I_list.append(I_func(t))
    elif family == "Triangle":
        for _ in range(num):
            value = rng.uniform(-1, 1)
            I_func = functions.TriangleCurrent(value * C)
            I_list.append(I_func(t))
    elif family == "CC":
        for _ in range(num):
            value = rng.uniform(-1, 1)
            I_func = functions.ConstantCurrent(value * C)
            I_list.append(I_func(t))

    elif family == "PLS":                                     # ← new branch
        for _ in range(num):
            I_func = PLSCurrent(C, t_max, rng)
            I_list.append(I_func(t))
            
    return I_list

In [7]:
def get_targets(I_samples, soc_stack=[0.5], Dan_stack=[3.3e-14], Dca_stack=[4.0e-15]):
    """
    For each current function in I_samples, set the battery parameters,
    run the simulation, and extract the target and initial concentrations.
    """
    set_targets_anode = []
    set_targets_cathode = []
    set_c0_anode = []
    set_c0_cathode = []
    set_currents = []
    set_socs = []
    set_Dan = []
    set_Dca = []

    # print(f"Running simulation for SoC={soc_stack}, Dan={Dan_stack}, Dca={Dca_stack}")

    for I_func, soc, Dan, Dca in zip(I_samples, soc_stack, Dan_stack, Dca_stack):
        #print(f"Running simulation for SoC={soc}, Dan={Dan}, Dca={Dca}")
        
        params_local = pybamm.ParameterValues("Prada2013")
        params_local["Current function [A]"] = pybamm.Interpolant(t, -1. * I_func, pybamm.t)
        params_local["Negative particle diffusivity [m2.s-1]"] = 10 ** Dan[0]
        params_local["Positive particle diffusivity [m2.s-1]"] = 10 ** Dca[0]
        sim = pybamm.Simulation(spm, parameter_values=params_local)
        sol = sim.solve(initial_soc=soc[0], t_eval=t)

        c0_anode = sol["Negative particle concentration"].entries[:, 0, 0]
        cn_target_anode = sol["Negative particle concentration"].entries[:, 0, :]
        c0_cathode = sol["Positive particle concentration"].entries[:, 0, 0]
        cn_target_cathode = sol["Positive particle concentration"].entries[:, 0, :]
        
        set_targets_anode.append(cn_target_anode)
        set_c0_anode.append(c0_anode)
        set_targets_cathode.append(cn_target_cathode)
        set_c0_cathode.append(c0_cathode)
        set_currents.append(I_func)

        set_socs.append(soc[0])
        set_Dan.append(Dan[0])
        set_Dca.append(Dca[0])
    
    return (set_targets_anode, set_c0_anode,
            set_targets_cathode, set_c0_cathode,
            set_currents, set_socs, set_Dan, set_Dca)

In [8]:
def save_data(family, soclala, Dan, Dca, data):
    """
    Save the generated data to a file.
    
    family: current profile family ("CC", "GRF", or "Triangle")
    soc: list of SOC levels
    Dan: list of negative particle diffusivities
    Dca: list of positive particle diffusivities
    data: tuple containing the generated data arrays
    """
    total_samples = len(data[0])
    # filename = f"data/{family}_{total_samples}.npz"
    filename = f"data/paperplots/{family}_{total_samples}.npz"
    cn_anode, c0_anode, cn_cathode, c0_cathode, current, soc, Dan, Dca = data

    np.savez(
        filename,
        soc=soc,
        Dan=Dan,
        Dca=Dca,
        cn_anode=cn_anode,
        c0_anode=c0_anode,
        cn_cathode=cn_cathode,
        c0_cathode=c0_cathode,
        current=current,
    )

    print(f"Saved train data to data/{family}_data.npz")

In [9]:
def run_batch(soc_chunk, Dan_chunk, Dca_chunk, child_seed, family="CC"):
    """
    Parameters
    ----------
    idx        : 1-D ndarray
        Row indices in `design` owned by this worker.
    child_seed : numpy.random.SeedSequence
        Unique SeedSequence for reproducible, independent randomness.
    Returns
    -------
    tuple_of_arrays  (or whatever `get_targets` produces)
    """
    # ---- 1) local RNG for current profiles ---------------------------------
    rng = np.random.default_rng(child_seed)

    # ---- 3) generate a matching number of currents -------------------------
    I_samples  = generate_data(rng, soc_chunk.shape[0], family=family)

    # ---- 4) run the expensive deterministic simulation --------------------
    result     = get_targets(I_samples, soc_chunk, Dan_chunk, Dca_chunk)

    return result

In [10]:
def generate_data_multiproccess(payload, n_workers = 48):

    assert n_workers == len(payload)

    with mp.Pool(n_workers) as pool:
        data_parts = pool.starmap(run_batch, payload)     # list of tuples/arrays

    (
    targets_anode_parts,
    c0_anode_parts,
    targets_cathode_parts,
    c0_cathode_parts,
    currents_parts,
    socs_parts,
    Dan_parts,
    Dca_parts,
    ) = zip(*data_parts)          # gives five shorter lists/tuples

    # ── 2. concatenate each list ────────────────────────────────
    targets_anode_all   = np.concatenate(targets_anode_parts,   axis=0)
    c0_anode_all        = np.concatenate(c0_anode_parts,        axis=0)
    targets_cathode_all = np.concatenate(targets_cathode_parts, axis=0)
    c0_cathode_all      = np.concatenate(c0_cathode_parts,      axis=0)
    currents_all        = np.concatenate(currents_parts,        axis=0)
    socs_all            = np.concatenate(socs_parts,            axis=0)
    Dan_all             = np.concatenate(Dan_parts,             axis=0)
    Dca_all             = np.concatenate(Dca_parts,             axis=0)

    # Bundle them back into one tuple (or dict)
    data = (
        targets_anode_all,
        c0_anode_all,
        targets_cathode_all,
        c0_cathode_all,
        currents_all,
        socs_all,
        Dan_all,
        Dca_all,
    )
    return data

In [11]:
soc_levels = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

n_workers = 40
families = ["GRF"]
samples_per_soc = 3000

# N_train = len(soc_levels) * samples_per_soc * num_train
# N_test = len(soc_levels) * samples_per_soc * num_test

# N_total = N_train + N_test

N_total = len(soc_levels) * samples_per_soc

# lower_bound_log_diffusion = -15.
# upper_bound_log_diffusion = -12.

lower_bound_log_diffusion = -18.
upper_bound_log_diffusion = -14.

seed = 42
rng = np.random.default_rng(seed)

master_ss   = np.random.SeedSequence(seed)
child_ss    = master_ss.spawn(n_workers)


sample_space = get_soc_and_diffusivities_sobol(
    soc_levels=soc_levels, total_samples=N_total,
    log_lower=lower_bound_log_diffusion, log_upper=upper_bound_log_diffusion, rng=rng)

soc_stack, Dan_stack, Dca_stack = np.split(sample_space, 3, axis=1)

indices   = np.array_split(np.arange(N_total), n_workers) 
# Create the SPM model and remove events if needed
spm = pybamm.lithium_ion.SPM()
spm.events = []

# Global battery parameters and time discretization:
params_bat = pybamm.ParameterValues("Prada2013")

### To enforce constant Dan and Dca across all samples, we can use the first sample:
# Dan_stack = np.full((N_total, 1), np.log10(params_bat["Negative particle diffusivity [m2.s-1]"]))
# Dca_stack = np.full((N_total, 1), np.log10(params_bat["Positive particle diffusivity [m2.s-1]"]))

C = params_bat["Nominal cell capacity [A.h]"]
t_max = 3600
num_samples_I = 75
num_samples_c0 = 20
t = np.linspace(0, t_max, num_samples_I)
r = np.linspace(0, 1, num_samples_c0)

for family in families:
    print(f"Processing family: {family}")

    payload = [
        (
            # indices[i],                           # idx
            soc_stack[indices[i]],                # soc_chunk
            Dan_stack[indices[i]],                # Dan_chunk
            Dca_stack[indices[i]],                # Dca_chunk
            int(child_ss[i].generate_state(1)[0]), # child_seed
            family

        )
        for i in range(n_workers)
]

    data = generate_data_multiproccess(payload, n_workers=n_workers)
    print("Data generation complete.")
    save_data(family, soc_stack, Dan_stack, Dca_stack, data)
    print(f"Saved data for family: {family}")
    print(f"Finished processing family: {family}")

Processing family: GRF


/home/dlu-apa/.pyenv/versions/3.11.4/lib/python3.11/site-packages/scipy/stats/_qmc.py:958: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  sample = self._random(n, workers=workers)


Data generation complete.
Saved train data to data/GRF_data.npz
Saved data for family: GRF
Finished processing family: GRF
